#  Zameen.com Property Preprocessing Pipeline

This notebook contains the complete data cleaning, parsing, and feature engineering pipeline for raw property crawled listings. It standardizes mixed currency scales (Lakh/Crore), land units (Kanal/Marla), target-encodes high-cardinality locations, and dummy-encodes categorical indicators.

## 1. Import Essential Libraries

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple

## 2. Text Parsing and Standardization Engines
We parse currency scales into absolute PKR and convert regional units (Kanal/Marla/Sq Yd) into uniform Square Yards.

In [2]:
def parse_price(raw_price_string: str) -> float:
    if not raw_price_string or not isinstance(raw_price_string, str):
        return 0.0
    raw_price = raw_price_string.lower().strip()
    
    if 'crore' in raw_price:
        components = raw_price.split('crore')
        crore_matches = re.findall(r'[\d\.]+', components[0])
        crore_val = float(crore_matches[0]) if crore_matches else 0.0
        lakh_val = 0.0
        if len(components) > 1 and 'lakh' in components[1]:
            lakh_matches = re.findall(r'[\d\.]+', components[1])
            lakh_val = float(lakh_matches[0]) if lakh_matches else 0.0
        return (crore_val * 10000000.0) + (lakh_val * 100000.0)
    
    if 'lakh' in raw_price:
        lakh_matches = re.findall(r'[\d\.]+', raw_price)
        parsed_val = float(lakh_matches[0]) if lakh_matches else 0.0
        return parsed_val * 100000.0
        
    if 'million' in raw_price:
        million_matches = re.findall(r'[\d\.]+', raw_price)
        parsed_val = float(million_matches[0]) if million_matches else 0.0
        return parsed_val * 1000000.0
        
    if 'thousand' in raw_price:
        thousand_matches = re.findall(r'[\d\.]+', raw_price)
        parsed_val = float(thousand_matches[0]) if thousand_matches else 0.0
        return parsed_val * 1000.0
        
    regex_numeric_matches = re.findall(r'[\d\.]+', raw_price)
    if regex_numeric_matches:
        return float(regex_numeric_matches[0])
    return 0.0

def parse_area(raw_area_string: str) -> float:
    if not raw_area_string or not isinstance(raw_area_string, str):
        return 0.0
    raw_area = raw_area_string.lower().strip()
    
    if 'kanal' in raw_area:
        kanal_matches = re.findall(r'[\d\.]+', raw_area)
        parsed_val = float(kanal_matches[0]) if kanal_matches else 0.0
        return parsed_val * 4500.0
        
    if 'marla' in raw_area:
        marla_matches = re.findall(r'[\d\.]+', raw_area)
        parsed_val = float(marla_matches[0]) if marla_matches else 0.0
        return parsed_val * 225.0
        
    if 'sq. yd.' in raw_area or 'sq.yd.' in raw_area or 'square yards' in raw_area:
        yard_matches = re.findall(r'[\d\.]+', raw_area)
        parsed_val = float(yard_matches[0]) if yard_matches else 0.0
        return parsed_val * 9.0
        
    regex_numeric_matches = re.findall(r'[\d\.]+', raw_area)
    if regex_numeric_matches:
        return float(regex_numeric_matches[0])
    return 0.0

## 3. Imputation and Cleaners
Filters duplicates, coerces numeric fields, checks valid construction years, and imputes missing data with median statistics.

In [3]:
def clean_missing_and_duplicates(raw_listings_dataframe: pd.DataFrame) -> pd.DataFrame:
    cleaned_listings_dataframe = raw_listings_dataframe.copy()
    cleaned_listings_dataframe = cleaned_listings_dataframe.drop_duplicates()
    
    target_numeric_columns = [
        'Bedrooms', 'Bathrooms', 'Built in year', 'Parking space', 
        'Servant Quarters', 'Store rooms', 'Kitchens', 'Drawing Rooms'
    ]
    
    for column_name in target_numeric_columns:
        if column_name in cleaned_listings_dataframe.columns:
            cleaned_listings_dataframe[column_name] = pd.to_numeric(
                cleaned_listings_dataframe[column_name], errors='coerce'
            )
            
    if 'Built in year' in cleaned_listings_dataframe.columns:
        invalid_years_mask = (
            (cleaned_listings_dataframe['Built in year'] < 1900) | 
            (cleaned_listings_dataframe['Built in year'] > 2026)
        )
        cleaned_listings_dataframe.loc[invalid_years_mask, 'Built in year'] = np.nan
            
    default_impute_medians = {
        'Bedrooms': 3.0,
        'Bathrooms': 3.0,
        'Built in year': 2018.0,
        'Parking space': 1.0,
        'Servant Quarters': 0.0,
        'Store rooms': 0.0,
        'Kitchens': 1.0,
        'Drawing Rooms': 1.0
    }
    
    for column_name, default_impute_value in default_impute_medians.items():
        if column_name in cleaned_listings_dataframe.columns:
            calculated_median_value = cleaned_listings_dataframe[column_name].median()
            final_fill_value = (
                calculated_median_value 
                if not pd.isna(calculated_median_value) 
                else default_impute_value
            )
            cleaned_listings_dataframe[column_name] = (
                cleaned_listings_dataframe[column_name].fillna(final_fill_value)
            )
            
    return cleaned_listings_dataframe

## 4. Encoding Structural Features
Applies target mean-encoding to Location and dummy one-hot encoding to Metropolitan City and Property Type.

In [4]:
def encode_categorical(property_dataframe: pd.DataFrame, categorical_columns: List[str]) -> pd.DataFrame:
    present_categorical_columns = [
        col for col in categorical_columns 
        if col in property_dataframe.columns
    ]
    if not present_categorical_columns:
        return property_dataframe.copy()
    return pd.get_dummies(
        property_dataframe, 
        columns=present_categorical_columns, 
        drop_first=True, 
        dtype=float
    )

def encode_location(
    property_dataframe: pd.DataFrame, 
    location_column_name: str = 'Location', 
    target_price_column_name: str = 'Price'
) -> pd.DataFrame:
    encoded_property_dataframe = property_dataframe.copy()
    if location_column_name in encoded_property_dataframe.columns:
        if target_price_column_name in encoded_property_dataframe.columns:
            location_target_mean_prices = (
                encoded_property_dataframe
                .groupby(location_column_name)[target_price_column_name]
                .mean()
            )
            global_mean_property_price = (
                encoded_property_dataframe[target_price_column_name].mean()
            )
            encoded_property_dataframe[location_column_name] = (
                encoded_property_dataframe[location_column_name]
                .map(location_target_mean_prices)
                .fillna(global_mean_property_price)
            )
        else:
            encoded_property_dataframe[location_column_name] = (
                encoded_property_dataframe[location_column_name]
                .astype('category')
                .cat.codes
            )
    return encoded_property_dataframe

## 5. Execution & Validation
Runs the pipeline end-to-end and outputs standard preprocessed files.

In [5]:
raw_data_path = '../assets/raw_data.csv'
if not os.path.exists(raw_data_path):
    raw_data_path = 'assets/raw_data.csv'

print(f"Loading raw property dataset from: {raw_data_path}")
df_raw = pd.read_csv(raw_data_path)
print(f"Raw dataset shape: {df_raw.shape}")
display(df_raw.head())

df_processed = df_raw.copy()
df_processed['Price'] = df_processed['Price'].apply(parse_price)
df_processed['Area'] = df_processed['Area'].apply(parse_area)
df_processed = clean_missing_and_duplicates(df_processed)
df_processed = encode_location(df_processed)
df_processed = encode_categorical(df_processed, ['Property type', 'City'])
df_processed = df_processed.round(2)

print(f"Preprocessed dataset shape: {df_processed.shape}")
display(df_processed.head())

cleaned_data_path = 'assets/cleaned_data.csv'
os.makedirs(os.path.dirname(cleaned_data_path), exist_ok=True)
df_processed.to_csv(cleaned_data_path, index=False)
print(f"Cleaned features successfully written to: {cleaned_data_path}")

Loading raw property dataset from: ../assets/raw_data.csv
Raw dataset shape: (974, 13)


,Price,Area,City,Bedrooms,Bathrooms,Location,Property type,Built in year,Parking space,Servant Quarters,Store rooms,Kitchens,Drawing Rooms
0,17 Crore,NaN,Islamabad,9.0,10.0,"F-10, Islamabad",House,2015.0,7.0,2.0,2.0,3.0,1.0
1,26.5 Crore,NaN,Islamabad,4.0,5.0,"F-8, Islamabad",House,2010.0,5.0,1.0,1.0,1.0,1.0
2,14 Crore,NaN,Islamabad,7.0,7.0,"F-11, Islamabad",House,2012.0,3.0,2.0,1.0,NaN,1.0
3,42 Crore,NaN,Islamabad,6.0,6.0,"F-11, Islamabad",House,2026.0,6.0,4.0,2.0,2.0,1.0
4,13.9 Crore,NaN,Islamabad,5.0,6.0,"DHA Defence Phase 2, DHA Defence",House,2025.0,3.0,2.0,2.0,2.0,1.0


Preprocessed dataset shape: (965, 13)


,Price,Area,Bedrooms,Bathrooms,Location,Built in year,Parking space,Servant Quarters,Store rooms,Kitchens,Drawing Rooms,Property type_House,Property type_Penthouse
0,170000000.0,0.0,9.0,10.0,2.719583e+08,2015.0,7.0,2.0,2.0,3.0,1.0,1.0,0.0
1,265000000.0,0.0,4.0,5.0,3.711842e+08,2010.0,5.0,1.0,1.0,1.0,1.0,1.0,0.0
2,140000000.0,0.0,7.0,7.0,2.393333e+08,2012.0,3.0,2.0,1.0,2.0,1.0,1.0,0.0
3,420000000.0,0.0,6.0,6.0,2.393333e+08,2026.0,6.0,4.0,2.0,2.0,1.0,1.0,0.0
4,139000000.0,0.0,5.0,6.0,1.314392e+08,2025.0,3.0,2.0,2.0,2.0,1.0,1.0,0.0


Cleaned features successfully written to: assets/cleaned_data.csv
